In [ ]:
import os
import pandas as pd
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from kfirlib.comparisons import *

In [ ]:
from elsapy.elsclient import ElsClient
from elsapy.elsprofile import ElsAuthor, ElsAffil
from elsapy.elsdoc import FullDoc, AbsDoc
from elsapy.elssearch import ElsSearch

In [ ]:
client = ElsClient('YOUR_API_KEY')
client.inst_token = 'YOUR_API_KEY'

In [ ]:
text = """(TITLE-ABS-KEY(vaccin* OR immuniz* OR immunis* OR inoculat*))
AND
(TITLE-ABS-KEY(barrier* OR enabler* OR facilitator* OR obstacle* OR challenge*))
AND
(TITLE-ABS-KEY(HPV OR "human papillomavirus" OR "papillomavirus vaccin*" OR "cervical cancer vaccin*" OR MMR OR "measles mumps rubella" OR "measles vaccin*" OR "mumps vaccin*" OR "rubella vaccin*"))
AND
(TITLE-ABS-KEY("health system*" OR "healthcare system*" OR "health service*" OR "health services" OR "service delivery" OR "delivery of health care" OR "healthcare access" OR "access to care" OR "primary care" OR "health services demand" OR "health services accessibility"))
AND
(TITLE-ABS-KEY("ethnic group*" OR immigrant* OR refugee* OR underserved OR disadvantaged OR vulnerable OR minority OR minorities OR "hard-to-reach" OR marginalized))
AND
(PUBYEAR > 2009 AND PUBYEAR < 2023)
AND
(LIMIT-TO(LANGUAGE, "English"))
"""


In [ ]:
DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables'

In [ ]:
doc_srch = ElsSearch(text, 'scopus')
doc_srch.execute(client, count=1000, use_cursor=True)

# Print how many records and the first record
print("Found {} documents".format(len(doc_srch.results)))

In [ ]:
doc_srch.hasAllResults()

In [ ]:
doc_srch.results_df.head()

In [ ]:
def make_row(result):
    doc = AbsDoc(scp_id=result['dc:identifier'])
    doc.read(els_client=client)

    data = doc.data['coredata']
    row = {}
    row['journal'] = data['prism:publicationName']
    row['title'] = data['dc:title']
    row['authors'] = ', '.join([author['ce:indexed-name'] for author in doc.data['authors']['author']])
    row['date'] = data['prism:coverDate']
    row['abstract'] = data['dc:description']
    return row

In [ ]:
def get_doc_data(result):
    doc = AbsDoc(scp_id=result['dc:identifier'])
    doc.read(els_client=client)
    return doc.data

In [ ]:
payloads = doc_srch.results
max_workers = 20

In [ ]:
results = [None] * len(payloads)  # keep order if needed
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    future_to_idx = {ex.submit(get_doc_data, p): i for i, p in enumerate(payloads)}
    for fut in tqdm(as_completed(future_to_idx), total=len(future_to_idx), desc="RPC calls"):
        i = future_to_idx[fut]
        results[i] = fut.result()  # will raise if the RPC failed


In [ ]:
rows = []
bad = []

for doc_data in results:
    if not doc_data:
        continue
    if 'coredata' not in doc_data:
        bad.append(doc_data)
        continue
    data = doc_data['coredata']
    row = {}
    row['journal'] = data.get('prism:publicationName')
    row['title'] = data.get('dc:title')
    try:
        row['authors'] = ', '.join([author.get('ce:indexed-name') for author in doc_data['authors']['author']])
    except:
        row['authors'] = None
    row['date'] = data.get('prism:coverDate')
    row['abstract'] = data.get('dc:description')
    rows.append(row)

scopus_df = pd.DataFrame(rows)
scopus_df['date'] = pd.to_datetime(scopus_df['date'], yearfirst=True)
scopus_df.head(10)

In [ ]:
scopus_df['date'] = pd.to_datetime(scopus_df['date'], yearfirst=True)
scopus_df.head(10)

In [ ]:
len(scopus_df)

In [ ]:
scopus_df.isna().sum()

In [ ]:
scopus_df.to_excel(os.path.join(DIR, 'scopus.xlsx'), index=False)

In [ ]:
pubmed_df = pd.read_excel(os.path.join(DIR, 'pubmed_results_Jamalah_515 - gpt_title_abstract.xlsx'))

In [ ]:
pubmed_df.columns

In [ ]:
scopus_df.columns

In [ ]:
pubmed_df.publication_date.head()

In [ ]:
pubmed_df['publication_date'] = pd.to_datetime(pubmed_df['publication_date'])
pubmed_df['publication_date'].head()

In [ ]:
scopus_df.rename(columns={'date': 'publication_date'}, inplace=True)
scopus_df.head()

In [ ]:
scopus_df['year'] = scopus_df['publication_date'].dt.year
scopus_df.head()

In [ ]:
scopus_df['source'] = 'scopus'
pubmed_df['source'] = 'pubmed'
all_df = pd.concat([scopus_df, pubmed_df], axis=0)
all_df.head()

In [ ]:
all_df.tail()

In [ ]:
all_df['title_extract'] = all_df.title.apply(extract_letters)
print(len(all_df))
# keep=last bc we prefer pubmed bc it already has results.
all_df.drop_duplicates('title_extract', keep='last', inplace=True, ignore_index=True)
print(len(all_df))

In [ ]:
df_495 = pd.read_excel(os.path.join(DIR, '495.xlsx'))
df_495.columns

In [ ]:
df_495['title_extract'] = df_495.Title.apply(extract_letters)
all_df['in_495'] = all_df.title_extract.isin(df_495['title_extract'])
all_df.groupby('source')['in_495'].sum()

In [ ]:
comps = {
    '86_for review': '86',
    'final30': '30',
    'prescreaned1501': '1501',
}
for name, short_name in comps.items():
    df = pd.read_excel(os.path.join(DIR, name + '.xlsx'))
    titles = df['title'] if 'title' in df.columns else df['Title']
    df['title_extract'] = titles.apply(extract_letters)
    all_df[f'in_{short_name}'] = all_df.title_extract.isin(df.title_extract)
all_df.columns

In [ ]:
all_df.to_excel(os.path.join(DIR, 'scopus_and_pubmed.xlsx'), index=False)

## Old stuff

In [ ]:
for result in doc_srch.results:
    print(f'{result["prism:coverDate"]}\t{result["dc:creator"]}\n{result["dc:title"]}')

In [ ]:
doc_srch.num_res

In [ ]:
doc.id

In [ ]:
doc.title

In [ ]:
result = doc_srch.results[0]
doc = FullDoc(sd_pii=result['pii'])
doc.client = client
doc.read()

In [ ]:
data = doc.data['coredata']
print('Journal\t\t', data['prism:publicationName'])
print('Title\t\t', data['dc:title'])
authors = data['dc:creator']
authors = [a['$'] for a in authors]
authors_string = '; '.join(authors)
print('Authors\t\t', authors_string)
print('Date\t\t', data['prism:coverDate'])
print('Abstract:\n', data['dc:description'])


In [ ]:
result = doc_srch.results[2]
doc = AbsDoc(scp_id=result['dc:identifier'])
doc.read(els_client=client)

In [ ]:
data = doc.data['coredata']
print('Journal\t\t', data['prism:publicationName'])
print('Title\t\t', data['dc:title'])
authors_string = ', '.join([author['ce:indexed-name'] for author in doc.data['authors']['author']])
print('Authors\t\t', authors_string)
print('Date\t\t', data['prism:coverDate'])
print('Abstract:\n', data['dc:description'])
